# PANTANAL-1 M11 — Non-Zero Baseline Notebook

This is an **m11 nonzero baseline notebook**. It can produce either:

- **REAL_SAMPLE_NONZERO_BASELINE** — uniform epsilon (default `0.001`) on all class columns using real Kaggle `sample_submission.csv`, writing `/kaggle/working/submission.csv` when that file is found.
- **SYNTHETIC_FALLBACK_ONLY** — M01/M10-style synthetic uniform-ε smoke when no real sample file exists.

**Non-claims:**

- Uniform epsilon baseline only; does not prove model inference or competitive quality.
- Does not prove leaderboard submission, score, or score improvement unless Kaggle shows a score and it is recorded in evidence.
- Does not prove audio understanding or trained model inference.
- Does not prove CPU 90-minute scoring runtime unless commit/submit-mode evidence is recorded.
- Does not prove Kaggle commit/submit-mode execution unless explicitly run and documented.
- Synthetic fallback does not prove real `sample_submission.csv` compatibility.

Record public score only if shown by Kaggle. Prior all-zero baseline public score was **0.500** (pipeline acceptance, not model quality).

In [ ]:
import os
import platform
import sys
import time
from pathlib import Path

EPSILON = 0.001

_m11_t0 = time.perf_counter()

print("=== PANTANAL-1 M11 Non-Zero Baseline Debug ===")
print("Python:", sys.version)
print("Platform:", platform.platform())
print("cwd:", Path.cwd())
print("sys.path first entries:", sys.path[:10])
print("KAGGLE_KERNEL_RUN_TYPE:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
print("KAGGLE_URL_BASE:", os.environ.get("KAGGLE_URL_BASE"))
print("EPSILON:", EPSILON)

for path in [Path("/kaggle"), Path("/kaggle/input"), Path("/kaggle/working"), Path("tmp")]:
    print(f"{path}: exists={path.exists()}")
    if path.exists():
        try:
            print(f"{path} children:", [p.name for p in list(path.iterdir())[:20]])
        except Exception as exc:
            print(f"{path} listing failed:", repr(exc))

In [ ]:
from pathlib import Path

try:
    from pantanal_1.kaggle_paths import (
        all_sample_submission_candidates,
        find_sample_submission_path,
        is_kaggle_environment,
    )
    from pantanal_1.nonzero_baseline import (
        build_synthetic_nonzero_fieldnames_and_rows,
        build_uniform_nonzero_rows,
        validate_nonzero_sample_rows,
        write_nonzero_submission_csv,
    )
    from pantanal_1.sample_baseline import (
        class_columns_from_fieldnames,
        read_sample_submission_csv,
    )

    SOURCE = "pantanal_1 package import"
except ModuleNotFoundError as exc:
    print("Package import failed:", repr(exc))
    print("Falling back to inline M11 uniform-epsilon implementation.")
    import csv

    SOURCE = "inline fallback"

    EXPLICIT_CANDIDATES = (
        Path("/kaggle/input/competitions/birdclef-2026/sample_submission.csv"),
        Path("/kaggle/input/birdclef-2026/sample_submission.csv"),
    )

    def is_kaggle_environment():
        return bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE")) or Path("/kaggle").exists()

    def discover_sample_submission_paths(root=Path("/kaggle/input")):
        if not root.exists():
            return ()
        found = []
        for path in sorted(root.rglob("sample_submission.csv")):
            if path.is_file():
                found.append(path)
        return tuple(found)

    def all_sample_submission_candidates():
        explicit = list(EXPLICIT_CANDIDATES)
        discovered = list(discover_sample_submission_paths())
        seen = set()
        ordered = []
        for p in explicit + discovered:
            if p not in seen:
                seen.add(p)
                ordered.append(p)
        return tuple(ordered)

    def find_sample_submission_path():
        for path in all_sample_submission_candidates():
            if path.is_file():
                return path
        return None

    def read_sample_submission_csv(sample_path):
        with sample_path.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            fieldnames = list(reader.fieldnames)
            rows = [dict(row) for row in reader]
        return fieldnames, rows

    def class_columns_from_fieldnames(fieldnames):
        if "row_id" not in fieldnames:
            raise ValueError("missing row_id")
        cols = [n for n in fieldnames if n != "row_id"]
        if not cols:
            raise ValueError("no class columns")
        return cols

    def _format_probability_string(epsilon):
        value = float(epsilon)
        if value <= 0.0 or value > 1.0:
            raise ValueError(f"epsilon out of range: {value}")
        text = f"{value:g}"
        if "." not in text and "e" not in text.lower():
            text = f"{text}.0"
        return text

    def build_uniform_nonzero_rows(sample_rows, fieldnames, epsilon=0.001):
        prob_str = _format_probability_string(epsilon)
        names = list(fieldnames)
        out = []
        for row in sample_rows:
            out.append({name: row["row_id"] if name == "row_id" else prob_str for name in names})
        return out

    def validate_nonzero_sample_rows(rows, fieldnames, epsilon=0.001):
        expected = _format_probability_string(epsilon)
        class_cols = class_columns_from_fieldnames(fieldnames)
        saw_nonzero = False
        for idx, row in enumerate(rows):
            if list(row.keys()) != list(fieldnames):
                raise ValueError(f"row {idx} column order mismatch")
            for col in class_cols:
                if float(row[col]) != 0.0:
                    saw_nonzero = True
                if row[col] != expected:
                    raise ValueError(f"row {idx} {col} expected {expected}")
        if not saw_nonzero:
            raise ValueError("no non-zero class values")

    def write_nonzero_submission_csv(rows, output_path, fieldnames, epsilon=0.001):
        validate_nonzero_sample_rows(rows, fieldnames, epsilon)
        output_path = Path(output_path)
        output_path.parent.mkdir(parents=True, exist_ok=True)
        with output_path.open("w", newline="", encoding="utf-8") as handle:
            writer = csv.DictWriter(handle, fieldnames=list(fieldnames))
            writer.writeheader()
            writer.writerows(rows)
        return output_path

    SYNTHETIC_CLASS_LABELS = tuple(f"synthetic_class_{idx:03d}" for idx in range(234))
    SYNTHETIC_SOUNDSCAPE_STEMS = (
        "BC2026_Test_0001_S05_20250227_010002",
        "BC2026_Test_0002_S05_20250227_010003",
    )

    def build_segment_end_times(duration_seconds=60, window_seconds=5):
        return list(range(window_seconds, duration_seconds + 1, window_seconds))

    def build_row_id(stem, end_time):
        return f"{stem}_{end_time}"

    def build_synthetic_nonzero_fieldnames_and_rows(epsilon=0.001):
        fieldnames = ["row_id", *SYNTHETIC_CLASS_LABELS]
        sample_rows = []
        for stem in SYNTHETIC_SOUNDSCAPE_STEMS:
            for end_time in build_segment_end_times():
                sample_rows.append({"row_id": build_row_id(stem, end_time)})
        rows = build_uniform_nonzero_rows(sample_rows, fieldnames, epsilon=epsilon)
        return fieldnames, rows


print("Import/bootstrap complete. SOURCE =", SOURCE)
print("is_kaggle_environment:", is_kaggle_environment())

candidates = all_sample_submission_candidates()
print("sample_submission.csv candidates (count):", len(candidates))
for cand in candidates:
    print("  candidate:", cand, "exists=", cand.is_file())

selected_sample = find_sample_submission_path()
print("selected sample submission path:", selected_sample)

In [ ]:
if selected_sample is not None:
    BASELINE_MODE = "REAL_SAMPLE_NONZERO_BASELINE"
    fieldnames, sample_rows = read_sample_submission_csv(selected_sample)
    class_cols = class_columns_from_fieldnames(fieldnames)
    rows = build_uniform_nonzero_rows(sample_rows, fieldnames, epsilon=EPSILON)
    output_path = Path("/kaggle/working/submission.csv")
    print("mode:", BASELINE_MODE)
    print("sample path:", selected_sample)
    print("epsilon:", EPSILON)
    print("class column count:", len(class_cols))
    print("row count from sample:", len(sample_rows))
else:
    BASELINE_MODE = "SYNTHETIC_FALLBACK_ONLY"
    fieldnames, rows = build_synthetic_nonzero_fieldnames_and_rows(epsilon=EPSILON)
    output_path = Path("tmp/submissions/m11_synthetic_nonzero_baseline.csv")
    class_cols = [n for n in fieldnames if n != "row_id"]
    print("mode:", BASELINE_MODE)
    print("epsilon:", EPSILON)
    print("synthetic class count:", len(class_cols))

print("output path:", output_path)
print("row count:", len(rows))
print("column count:", len(fieldnames))

if class_cols:
    first_class = class_cols[0]
    print("non-zero check (first class col):", rows[0][first_class], "!= 0")
    assert float(rows[0][first_class]) != 0.0

In [ ]:
written = write_nonzero_submission_csv(rows, output_path, fieldnames, epsilon=EPSILON)

print("wrote:", written)
print("exists:", written.exists())
print("size bytes:", written.stat().st_size if written.exists() else None)
print("first row_id:", rows[0]["row_id"])
print("last row_id:", rows[-1]["row_id"])
print("header preview:", list(fieldnames)[:8], "...")
print("epsilon:", EPSILON)

with written.open("r", encoding="utf-8") as handle:
    for idx in range(3):
        line = handle.readline().rstrip()
        print(f"csv line {idx}:", line[:500])

_m11_elapsed = time.perf_counter() - _m11_t0
print("runtime seconds (notebook path):", round(_m11_elapsed, 3))

In [ ]:
print("=== M11 non-zero baseline claim summary ===")
print("BASELINE_MODE:", BASELINE_MODE)
print("EPSILON:", EPSILON)

if BASELINE_MODE == "SYNTHETIC_FALLBACK_ONLY":
    print("No real Kaggle submission path was proven.")
    print("Output:", output_path)
    print("Synthetic fallback only; does not prove real sample_submission.csv on Kaggle.")
else:
    print("Generated /kaggle/working/submission.csv from real sample_submission.csv schema.")
    print("Uniform epsilon baseline only; does not prove model inference or model quality.")
    print("Does not prove leaderboard submission or score unless submitted and recorded.")
    print("Record public score only if shown by Kaggle.")
    print("Output:", output_path)

print("This notebook does not prove score improvement over the prior 0.500 zero baseline.")